# 进阶教程（三）：Runtime 状态更新与时间旅行

> LangGraph 把每一步状态都存成 checkpoint。本讲用好这份"存档"：
> 人工干预（update_state）、历史回放（get_state_history）、分叉重跑。

## 本讲内容
1. 观察状态：`get_state` / `get_state_history`
2. 人工干预：`update_state` + 续跑
3. 时间旅行：从任意历史检查点分叉重放

# 0. 环境准备与运行说明

**前置要求：**
- 根目录 `.env` 已配置 `DEEPSEEK_API_KEY`（本教程用真实 DeepSeek，无本地降级）
- 已安装：`langchain>=1.3`、`langgraph>=1.2`、`langchain-deepseek`、`python-dotenv`
- 使用本地 `bge-small-zh-v1.5` 嵌入的章节首次运行会下载模型（约 100MB，走 hf-mirror 镜像）

**运行说明：**
- 按 cell 顺序执行；除标注外，每个示例消耗少量 API 额度（单次 < 0.01 元量级）
- 本教程面向已学完 `langchain_tutorial/` 与 `langgraph_tutorial/` 基础篇的开发者
- 涉及导入路径的坑（如 `create_agent` 在 `langchain.agents`）已在 FAQ 中汇总

In [1]:

# ========== 0. 初始化（每个 notebook 第一格） ==========
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# HF 镜像必须先于任何 langchain/huggingface 导入设置（详见 rag_qa_project FAQ）
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

ROOT = Path.cwd().parent  # advanced_tutorial 的上一级 = 项目根目录
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

assert os.getenv("DEEPSEEK_API_KEY"), "请先在根目录 .env 配置 DEEPSEEK_API_KEY"

# 真实 LLM：DeepSeek（本教程要求真实模型）
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat", temperature=0.2)
print("模型就绪:", model.__class__.__name__)


模型就绪: ChatDeepSeek


## 1. 观察状态：快照与历史

先建一个带 checkpointer 的多步图，跑完后"翻监控"：

In [2]:

from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class S(TypedDict):
    messages: Annotated[list, lambda o, n: o + n]
    draft: str

def step_a(state: S):
    return {"draft": "初稿 v1"}

def step_b(state: S):
    r = model.invoke(f"把这句话润色得更优雅：{state['draft']}")
    return {"draft": r.content}

g = StateGraph(S)
g.add_node("step_a", step_a)
g.add_node("step_b", step_b)
g.add_edge(START, "step_a")
g.add_edge("step_a", "step_b")
g.add_edge("step_b", END)
app = g.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "demo-1"}}
app.invoke({"messages": [], "draft": ""}, config=cfg)

# 当前快照
snap = app.get_state(cfg)
print("当前 draft:", snap.values["draft"][:50])
print("下一节点:", snap.next)

当前 draft: 好的，这里有几个不同风格的润色方案，您可以根据喜好和语境选择。

**1. 简洁正式版**
此版本最
下一节点: ()


In [3]:

# 完整历史：每次节点执行都是一条 checkpoint
for i, h in enumerate(app.get_state_history(cfg)):
    print(f"#{i} next={h.next} draft={str(h.values.get('draft', ''))[:40]!r}")

#0 next=() draft='好的，这里有几个不同风格的润色方案，您可以根据喜好和语境选择。\n\n**1. 简洁'
#1 next=('step_b',) draft='初稿 v1'
#2 next=('step_a',) draft=''
#3 next=('__start__',) draft=''


**预期输出**（大意）：历史按时间倒序排列（最新在前），
能看到每一步之后的状态切片——这就是"存档"。

## 2. 人工干预：update_state

运行中/运行后直接改状态。`as_node` 声明"以谁的身份写入"，
决定了下一步从哪里继续——**这是把人类插入工作流的标准姿势**：

In [5]:

# 模拟场景：step_a 产出的初稿被人工推翻
app.update_state(
    cfg,
    {"draft": "人工指定的初稿：让 LangGraph 成为 Agent 时代的操作系统"},
    as_node="step_a",   # 冒充 step_a 写入 → 下一步将从 step_b 继续
)
snap = app.get_state(cfg)
print("干预后 draft:", snap.values["draft"][:40])
print("下一步将执行:", snap.next)   # ('step_b',)

# 传 None 续跑：从当前检查点继续执行剩余节点
result = app.invoke(None, config=cfg)
print("续跑结果:", result["draft"][:600])

干预后 draft: 人工指定的初稿：让 LangGraph 成为 Agent 时代的操作系统
下一步将执行: ('step_b',)
续跑结果: 好的，这句话本身已经非常精炼有力。润色的目标是在不改变其核心冲击力的前提下，增加文采、深度或画面感。

这里有几个不同风格的版本，供您选择：

---

### **方案一：简洁有力版**

> **让 LangGraph 成为 Agent 时代的操作系统。**

这个版本最接近原句，但通过调整语序和用词，使其更具宣言感和确定性。将“人工指定的初稿”去掉，直接陈述目标，显得更加自信和果断。

---

### **方案二：诗意比喻版**

> **如果说 Agent 是新时代的公民，那么 LangGraph 便是孕育他们的数字国度。**

这个版本将“操作系统”升华为“数字国度”，将“Agent”比作“公民”，画面感极强，赋予了技术以生命和温度，暗示LangGraph不仅是工具，更是生态和文明的基石。

---

### **方案三：哲学思辨版**

> **Agent 时代呼唤其基石，而 LangGraph，正是为承载万智而生的操作系统。**

这个版本采用“呼唤”与“承载”的动词，营造出一种历史使命感和必然性。将“Agent”称为“万智”，提升了其智能的维度，显得更加宏大和深刻。

---

### **方案四：科技愿景版**

> **LangGraph，为 Agent 时代构筑的操作系统。**

这个版本最为凝练，将核心信息前置，用“构筑”一词，强调了其从无到有的创造性和工


**要点**：
- `update_state` 不会执行任何节点，只写状态并追加 checkpoint
- `invoke(None, config)` = "从上次停的地方继续跑"，配合 update_state 即"改档再玩"

## 3. 时间旅行：从历史检查点分叉

拿到任意历史 checkpoint_id，指定新的 thread_id 重放——
原线程不受影响，天然支持"如果当时……"的 A/B 实验：

In [6]:

# 找到"step_a 刚执行完、step_b 还没跑"的历史检查点
history = list(app.get_state_history(cfg))
old = [h for h in history if h.next == ("step_b",)][0]
print("回放点 draft:", old.values["draft"][:30], "| next:", old.next)

# ---- 方式一：同线程重放（直接用历史快照的 config）----
replay = app.invoke(None, old.config)
print("重放结果:", replay["draft"][:50])

# ---- 方式二：新线程分叉（原线程不受影响，可做 A/B 实验）----
fork_cfg = {"configurable": {
    "thread_id": "fork-1",                                        # 新线程
    "checkpoint_id": old.config["configurable"]["checkpoint_id"], # 从历史点开始
}}
# 先用 update_state 把该检查点落进新线程（可顺带修改状态）
app.update_state(fork_cfg, {"draft": "分叉修改稿：换个写法"}, as_node="step_a")
# 再在新线程续跑
fork_result = app.invoke(None, {"configurable": {"thread_id": "fork-1"}})
print("分叉结果:", fork_result["draft"][:50])
print("原线程 draft 不变:", app.get_state(cfg).values["draft"][:40])

回放点 draft: 人工指定的初稿：让 LangGraph 成为 Agent 时 | next: ('step_b',)
重放结果: 好的，这句话本身已经很有力，但我们可以从不同角度进行润色，让它更具文学性、技术深度或战略视野。

这
分叉结果: 好的，这里有几个不同风格的润色方案，您可以根据具体语境和偏好选择：

---

### **方案一：
原线程 draft 不变: 好的，这句话本身已经很有力，但我们可以从不同角度进行润色，让它更具文学性、技术深


**应用场景**：
| 场景 | 做法 |
|---|---|
| 调试回放 | 从出错前的检查点重跑，定位是哪一步引入的问题 |
| 人工修正 | update_state 改正某个字段后续跑，无需从头再来 |
| A/B 实验 | 同一检查点分叉多个线程，分别换参数/模型对比 |
| 生产审计 | get_state_history 即完整操作留痕 |

## 4. 常见问题（FAQ）

| 问题 | 原因 | 解决 |
|---|---|---|
| update_state 后状态没变 | 字段不在 State schema | TypedDict 里声明该字段 |
| `invoke(None)` 抛 GraphRecursionError | 图已到 END，None 触发重跑空转 | 先确认 `snap.next` 非空 |
| 时间旅行后消息重复累加 | messages 有 reducer | 分叉前理解 reducer 语义；必要时用 Remove 删除消息 |
| checkpoint_id 从哪来 | 不是手写的 | 从 `state.config["configurable"]["checkpoint_id"]` 读 |
| InMemorySaver 重启就丢 | 内存实现 | 生产换 SqliteSaver / PostgresSaver，API 完全一致 |